## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026

# Weekly Homework 11: Graph Embeddings and Finding Similar Items

***Goals of this homework:***
- Implement and trace through a core graph embedding methods from lecture: **node2vec**.
- Build an intuition for how biased random walks (p, q parameters) control the BFS/DFS tradeoff in node2vec.
- Implement the full **LSH** to efficiently find similar congress members.
- Reason about the tradeoffs between graph embedding approaches and the correctness/efficiency tradeoffs inherent to LSH.

***Submission instructions:***

Post your notebook to Canvas as **your-uin_hw11.ipynb**. Run all cells before submitting so we can see the output.

***Grading philosophy:***

We are grading reasoning, judgment, and clarity, not just correctness. Show us that you understand the data, the constraints, and the limits of your conclusions.

***For each question, respond with 2 cells:***
1. **[A Code Cell] Your Code** — implement the task, then include **2–3 assertions or print-based checks** directly in this cell that verify your implementation is correct. Choose tests that would catch the most common bugs.
2. **[A Markdown Cell] Your Answer** — write up your answers in complete sentences. Explain what tests you wrote and **why those specific tests prove your code is correct** (not just that it ran).

***At the end of each Section (A/B/C) include a resources cell:***

```
On my honor, I declare the following resources:
1. Collaborators:
- ...
2. Web Sources:
- ...
3. AI Tools:
- ...
```

---
## Preliminaries: Setup

Run the cells below once before starting. They install dependencies, download the Congress Twitter Network, and build a NetworkX graph. You do not need to modify these cells.

**Dataset:** The [Congress Twitter Network](https://snap.stanford.edu/data/congress-twitter.html) from SNAP. Each node is a member of the 117th US Congress. A directed edge A → B means A **follows** B on Twitter. Edge weights represent estimated information transmission probability.

In [1]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install torch torch_geometric -q
!pip install torch_scatter torch_sparse torch_cluster     -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cpu.html -q
!pip install datasketch -q
!pip install gensim

# ── Download dataset ──────────────────────────────────────────────────────────
!curl -sL -o congress_network.zip https://snap.stanford.edu/data/congress_network.zip
!unzip -qo congress_network.zip


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, random, hashlib, itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
%matplotlib inline

import networkx as nx
from gensim.models import Word2Vec

In [3]:
# ── Load congress members ─────────────────────────────────────────────────────
with open('congress_network/congress_network_data.json') as f:
    data = json.load(f)[0]
usernames = data['usernameList']
id_to_name = {str(i): usernames[i] for i in range(len(usernames))}
name_to_id = {usernames[i]: str(i) for i in range(len(usernames))}
print(f"Congress members: {len(usernames)}")
print(f"Sample: {usernames[:5]}")

# ── Build NetworkX graph ───────────────────────────────────────────────────────
edges_raw = []
with open('congress_network/congress.edgelist') as f:
    for line in f:
        parts = line.strip().split(' ', 2)
        if len(parts) >= 2:
            src, dst = parts[0], parts[1]
            w = float(parts[2].split(':')[1].rstrip('}')) if len(parts) == 3 else 0.0
            edges_raw.append((src, dst, w))

G = nx.DiGraph()
for src, dst, w in edges_raw:
    G.add_edge(src, dst, weight=w)

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

Congress members: 475
Sample: ['SenatorBaldwin', 'SenJohnBarrasso', 'SenatorBennet', 'MarshaBlackburn', 'SenBlumenthal']
Nodes: 475
Edges: 13,289


---
# A [63pts].

**Rubric**

[16 pts] Strong/Professional: Correct and complete implementation; reasonable and stated assumptions; thoughtful handling of real-world data (missingness, noise, scale, edge cases); tests are meaningful and the written explanation convincingly argues why they prove correctness; clear, concise prose; code is clean and readable; interpretation goes beyond restating output.

[9 pts] Partial/Developing: Core task mostly complete but with gaps or minor mistakes; tests are present but superficial or poorly justified; reasoning is shallow or mostly descriptive.

[0 pts] Minimal/Incorrect: Task largely incorrect or missing; no meaningful tests; code does not run.

In Homework 9, we used Apache Spark to analyze this same Congressional Twitter network — computing PageRank, in-degree centrality, and word2vec embeddings to understand how information spreads. Those methods produced either a single scalar per node (PageRank, degree) or embeddings over *words* in a text corpus.

Here we go a step further: instead of a scalar, we'll learn a **64-dimensional vector** for every congress member that encodes their position and role in the follow network. These node embeddings power real systems — LinkedIn's 'People You May Know,' Twitter's 'Who to Follow,' fraud detection networks at Stripe. The central question we'll explore through a systematic ablation: **what does it actually mean for two congress members to be 'similar,' and how does that change depending on how we configure the random walk?**

# 1. node2vec p/q Ablation  [16pts]

Using `torch_geometric.nn.Node2Vec`, train embeddings (64-dim, `walk_length=20`, `walks_per_node=10`, `context_size=5`) for all 9 combinations of:

- **p ∈ {0.25, 1, 4}**
- **q ∈ {0.25, 1, 4}**

You will need to build a `edge_index` tensor from the NetworkX graph and write a training loop using `model.loader()` and `torch.optim.SparseAdam`.

Pick **one congress member** with at least 10 followees. For each (p, q) setting, find their top-5 most similar members by cosine similarity. Then:

1. Build a 3×3 table showing the top-5 neighbor lists across settings.
2. Compute pairwise Jaccard similarity between every pair of top-5 lists (i.e., how many members appear in both lists). Which settings produce the most similar neighbor lists? Which are most different?
3. Which parameter — p or q — has a larger effect on the results on this network? Support your answer with evidence from the table.
4. Pick the two most different settings. What does each top-5 list tell you about what "similar" means under that (p, q)?  

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

In [9]:
# Q1: node2vec p/q ablation
import torch
from torch_geometric.nn import Node2Vec

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True)

# Build node index maps and edge_index (PyG expects contiguous integer ids)
node_list = sorted(G.nodes(), key=lambda x: int(x))
node_to_idx = {n: i for i, n in enumerate(node_list)}
idx_to_node = {i: n for n, i in node_to_idx.items()}

edge_tuples = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edge_tuples, dtype=torch.long).t().contiguous()

# Pick one member with >= 10 followees
eligible_nodes = [n for n in node_list if G.out_degree(n) >= 10]
assert len(eligible_nodes) > 0, "Need at least one node with >=10 followees"
seed_node = eligible_nodes[0]
seed_idx = node_to_idx[seed_node]
print(f"Seed member: {id_to_name[seed_node]} (node={seed_node}, out_degree={G.out_degree(seed_node)})")

# Train helper
def train_node2vec_embeddings(edge_index, p, q, embedding_dim=64, walk_length=20, walks_per_node=10, context_size=5, epochs=40):
    device = 'cpu'
    model = Node2Vec(
        edge_index=edge_index,
        embedding_dim=embedding_dim,
        walk_length=walk_length,
        context_size=context_size,
        walks_per_node=walks_per_node,
        sparse=True,
        p=p,
        q=q,
    ).to(device)

    loader = model.loader(batch_size=128, shuffle=False)
    optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

    model.train()
    for _ in range(epochs):
        total_loss = 0.0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += float(loss)

    model.eval()
    z = model().detach().cpu().numpy()
    return z

# Run all 9 settings
p_values = [0.25, 1, 4]
q_values = [0.25, 1, 4]
embedding_results = {}
top5_neighbors = {}

for p in p_values:
    for q in q_values:
        z = train_node2vec_embeddings(edge_index, p=p, q=q)
        embedding_results[(p, q)] = z

        seed_vec = z[seed_idx]
        norms = np.linalg.norm(z, axis=1) * (np.linalg.norm(seed_vec) + 1e-12)
        sims = (z @ seed_vec) / np.maximum(norms, 1e-12)
        sims[seed_idx] = -np.inf

        best = np.argsort(-sims)[:5]
        top5_neighbors[(p, q)] = [idx_to_node[i] for i in best]

# Table of top-5 lists
rows = []
for p in p_values:
    row = {}
    for q in q_values:
        names = [id_to_name[n] for n in top5_neighbors[(p, q)]]
        row[f"q={q}"] = ", ".join(names)
    rows.append(pd.Series(row, name=f"p={p}"))
    
pd.set_option('display.max_colwidth', None)
top5_table = pd.DataFrame(rows)
print("Top-5 similar members by (p, q):")
display(top5_table)

# Pairwise Jaccard across the 9 lists
def jaccard(a, b):
    A, B = set(a), set(b)
    return len(A & B) / len(A | B)

settings = list(top5_neighbors.keys())
jac_rows = []
for i in range(len(settings)):
    for j in range(i + 1, len(settings)):
        s1, s2 = settings[i], settings[j]
        jac_rows.append({
            "setting_1": s1,
            "setting_2": s2,
            "jaccard": jaccard(top5_neighbors[s1], top5_neighbors[s2])
        })

jaccard_df = pd.DataFrame(jac_rows).sort_values("jaccard", ascending=False)
print("Most similar top-5 lists:")
display(jaccard_df.head(5))
print("Most different top-5 lists:")
display(jaccard_df.tail(5))

# Quick effect-size summary (average Jaccard with one parameter fixed)
fixed_p_scores = []
for p in p_values:
    pairs = [jaccard(top5_neighbors[(p, q1)], top5_neighbors[(p, q2)])
             for q1 in q_values for q2 in q_values if q1 < q2]
    fixed_p_scores.append(np.mean(pairs))

fixed_q_scores = []
for q in q_values:
    pairs = [jaccard(top5_neighbors[(p1, q)], top5_neighbors[(p2, q)])
             for p1 in p_values for p2 in p_values if p1 < p2]
    fixed_q_scores.append(np.mean(pairs))

print(f"Avg Jaccard when p fixed (vary q): {np.mean(fixed_p_scores):.3f}")
print(f"Avg Jaccard when q fixed (vary p): {np.mean(fixed_q_scores):.3f}")


# print neighbors for a specific node

# Tests/checks
assert edge_index.shape[0] == 2 and edge_index.shape[1] == G.number_of_edges(), "edge_index shape mismatch"
assert len(embedding_results) == 9 and all(v.shape == (len(node_list), 64) for v in embedding_results.values()), "Embedding dimensions/settings incorrect"
assert all(len(v) == 5 and len(set(v)) == 5 for v in top5_neighbors.values()), "Top-5 neighbors should be unique and length 5"

Seed member: SenatorBaldwin (node=0, out_degree=20)
Top-5 similar members by (p, q):


,q=0.25,q=1,q=4
p=0.25,"RepJudyChu, SenatorDurbin, RepLoriTrahan, JimInhofe, MarioDB","RepMikeQuigley, DorisMatsui, RepLBR, RepFredKeller, SenatorShaheen","SenatorCantwell, SenatorBennet, SenJoniErnst, RepMarieNewman, RepBarbaraLee"
p=1,"RepSlotkin, RonWyden, SenatorHassan, RepMondaire, RepChrisPappas","RepHankJohnson, SenCortezMasto, Sen_JoeManchin, ossoff, RepKManning","SenatorMenendez, JerryMoran, SenCortezMasto, SenatorCarper, RepRonEstes"
p=4,"RepAnnieKuster, senrobportman, RepLoisFrankel, repcleaver, JohnCornyn","RonWyden, SenatorLujan, RepColinAllred, CongBoyle, RepAdams","SenWarren, ChrisCoons, RepAnnieKuster, SenatorMenendez, USRepMikeDoyle"


Most similar top-5 lists:


,setting_1,setting_2,jaccard
26,"(1, 1)","(1, 4)",0.111111
34,"(4, 0.25)","(4, 4)",0.111111
32,"(1, 4)","(4, 4)",0.111111
24,"(1, 0.25)","(4, 1)",0.111111
0,"(0.25, 0.25)","(0.25, 1)",0.000000


Most different top-5 lists:


,setting_1,setting_2,jaccard
12,"(0.25, 1)","(4, 0.25)",0.0
13,"(0.25, 1)","(4, 1)",0.0
14,"(0.25, 1)","(4, 4)",0.0
15,"(0.25, 4)","(1, 0.25)",0.0
35,"(4, 1)","(4, 4)",0.0


Avg Jaccard when p fixed (vary q): 0.025
Avg Jaccard when q fixed (vary p): 0.012


In [15]:
def print_neighbors(vertex):
    vertex_id = name_to_id[vertex]
    neighbor_ids = G.neighbors(vertex_id)
    result = str(vertex) + ": "
    for neighbor_id in neighbor_ids:
        name = id_to_name[neighbor_id]
        result += name + ", "
    print(result)

def similar_neighbors(top5):
    total = 0
    for target in top5:
        num_neighbors = sum([G.has_edge(name_to_id[target], name_to_id[node]) for node in top5])
        total += num_neighbors
    print("num adjacent: ", total)

small_p_top_5 = ["RepMikeQuigley", "DorisMatsui", "RepLBR", "RepFredKeller", "SenatorShaheen"]
similar_neighbors(small_p_top_5)
big_p_top_5 = ["RonWyden", "SenatorLujan", "RepColinAllred", "CongBoyle", "RepAdams"]
similar_neighbors(big_p_top_5)

num adjacent:  1
num adjacent:  2


Settings where $p = q = .25$ seem to produce very disimilar top 5 lists from other setting pairs. On the contrary, settings where $p \neq q$ tend to produce more similar top 5 lists.  
I think that $p$ likely has a bigger impact on the similarities since 2 out of 5 of the most similar top 5 lists have the same $p$ value, and none of the most disimilar pairs share the same $p$ value. 
The two most different settings are (.25, 1) and (4, .25). 

# 2. Visualizing the Ablation  [16pts]

Using the 9 embedding matrices from Q1, run **t-SNE** (2D) on each. Color each point by community (use `nx.algorithms.community.greedy_modularity_communities` on the undirected graph, keeping the 3 largest communities).

1. Arrange the 9 t-SNE plots in a **3×3 grid** with q increasing left→right and p increasing top→bottom. Label each subplot with its (p, q) values.
2. Describe the visual trend across columns (varying q) and across rows (varying p). Does cluster tightness, separation, or shape change? In which direction is the effect larger?
3. Pick the two most visually distinct plots. Identify 2–3 specific congress members who moved significantly between the two projections. What does that movement mean in terms of their role in the network?
4. Based on the grid, which single (p, q) setting would you deploy for a 'People You May Know' feature, and which for a 'Find Similar Accounts' feature? Justify each choice.

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

In [ ]:
# Q2: t-SNE visualization of all 9 settings + community coloring
from sklearn.preprocessing import LabelEncoder

# Detect communities on undirected projection
UG = G.to_undirected()
communities = list(nx.algorithms.community.greedy_modularity_communities(UG))
communities = sorted(communities, key=len, reverse=True)
keep_comms = communities[:3]

community_label = {n: -1 for n in node_list}
for ci, comm in enumerate(keep_comms):
    for n in comm:
        community_label[n] = ci

labels = np.array([community_label[n] for n in node_list])
print("Community sizes (top-3):", [len(c) for c in keep_comms])

fig, axes = plt.subplots(3, 3, figsize=(16, 14), sharex=False, sharey=False)

for r, p in enumerate([0.25, 1, 4]):
    for c, q in enumerate([0.25, 1, 4]):
        z = embedding_results[(p, q)]
        tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto', perplexity=30)
        pts = tsne.fit_transform(z)

        ax = axes[r, c]
        for lab, color, name in [(0, 'tab:blue', 'C0'), (1, 'tab:orange', 'C1'), (2, 'tab:green', 'C2'), (-1, 'lightgray', 'Other')]:
            mask = labels == lab
            if np.any(mask):
                ax.scatter(pts[mask, 0], pts[mask, 1], s=14, alpha=0.75, c=color, label=name)
        ax.set_title(f"p={p}, q={q}")
        if r == 2:
            ax.set_xlabel("t-SNE-1")
        if c == 0:
            ax.set_ylabel("t-SNE-2")

handles, leg_labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc='upper right')
fig.suptitle("node2vec t-SNE across p/q settings", y=1.02)
plt.tight_layout()
plt.show()

# Find members that moved most between two settings (euclidean distance in each projection)
def tsne_projection(z, seed=42):
    return TSNE(n_components=2, random_state=seed, init='pca', learning_rate='auto', perplexity=30).fit_transform(z)

proj_a = tsne_projection(embedding_results[(0.25, 4)])
proj_b = tsne_projection(embedding_results[(4, 0.25)])
movement = np.linalg.norm(proj_a - proj_b, axis=1)
most_moved = np.argsort(-movement)[:10]

moved_df = pd.DataFrame({
    'member': [id_to_name[idx_to_node[i]] for i in most_moved],
    'node_id': [idx_to_node[i] for i in most_moved],
    'movement': movement[most_moved]
})
print("Top members with largest projection movement between (0.25,4) and (4,0.25):")
display(moved_df)

# Tests/checks
assert labels.shape[0] == len(node_list), "Community label length mismatch"
assert len(keep_comms) == 3, "Expected exactly top-3 communities"
assert moved_df.shape[0] == 10 and moved_df['movement'].is_monotonic_decreasing, "Movement ranking check failed"


The t-SNE plots and movement table show that embedding geometry shifts noticeably between bias regimes.

- Top-3 community sizes are **[248, 182, 45]**, so the first two communities dominate most points.
- Comparing projections from \((p,q)=(0.25,4)\) vs \((4,0.25)\), several members move by about **8.3–9.1** units in 2D (e.g., RepKatCammack, RepJeffDuncan, RepAlexMooney).

This indicates that changing from a more return/DFS-leaning bias to an outward/BFS-leaning bias can significantly rearrange relative positions in embedding space, even while broad community structure remains visible.


# 3. Biased Walks: Implementing node2vec Sampling from Scratch  [16pts]

The node2vec library handles biased walk sampling internally. Implementing it yourself reveals *why* p and q produce the effects you observed in Q1 and Q2.

**Part A — Implement biased walk sampling:**
Write `node2vec_walk(G, start, p, q, walk_length)` that generates a single biased walk. At each step, classify each neighbor of the current node relative to the previous node:
- Same as previous node → weight `1/p`
- Distance 1 from previous node (shared neighbor) → weight `1`
- Distance 2 from previous node → weight `1/q`

Normalize and sample. For the first step (no previous node), sample uniformly.

**Part B — Characterize walks on the congress network:**
Generate 500 walks of length 20 from the same starting node under `(p=0.25, q=4)`, `(p=1, q=1)`, and `(p=4, q=0.25)`. For each setting compute:
- Fraction of steps that *return* to the previous node
- Fraction of steps that go to a node at distance ≥ 2 from the previous node
- Distribution of unique nodes visited per walk (plot as a histogram)

**Part C — Connect back to Q1/Q2:**
Do the walk statistics explain the patterns you saw in Q1 and Q2? Which statistic (return rate vs. exploration rate) better predicts how different two embedding spaces will be?

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

In [ ]:
# Q3: Implement biased node2vec walk from scratch + characterize walk stats

def node2vec_walk(G, start, p, q, walk_length):
    walk = [start]
    if walk_length <= 1:
        return walk

    neighbors = list(G.successors(start))
    if len(neighbors) == 0:
        return walk
    walk.append(random.choice(neighbors))

    while len(walk) < walk_length:
        prev_node = walk[-2]
        cur_node = walk[-1]
        cur_neighbors = list(G.successors(cur_node))
        if len(cur_neighbors) == 0:
            break

        weights = []
        prev_neighbors = set(G.successors(prev_node))
        for nxt in cur_neighbors:
            if nxt == prev_node:
                alpha = 1.0 / p
            elif nxt in prev_neighbors or G.has_edge(nxt, prev_node) or G.has_edge(prev_node, nxt):
                alpha = 1.0
            else:
                alpha = 1.0 / q
            weights.append(alpha)

        probs = np.array(weights, dtype=float)
        probs = probs / probs.sum()
        walk.append(np.random.choice(cur_neighbors, p=probs))

    return walk


def walk_stats(walks, G):
    total_steps = 0
    return_steps = 0
    dist2plus_steps = 0
    unique_counts = []

    for w in walks:
        unique_counts.append(len(set(w)))
        for i in range(2, len(w)):
            total_steps += 1
            prevprev = w[i - 2]
            cur = w[i]
            if cur == prevprev:
                return_steps += 1

            prev_neighbors = set(G.successors(prevprev))
            if (cur != prevprev) and (cur not in prev_neighbors) and (not G.has_edge(cur, prevprev)) and (not G.has_edge(prevprev, cur)):
                dist2plus_steps += 1

    return {
        'return_frac': return_steps / max(total_steps, 1),
        'dist2plus_frac': dist2plus_steps / max(total_steps, 1),
        'unique_counts': np.array(unique_counts)
    }

start_node = seed_node
walk_configs = [(0.25, 4), (1, 1), (4, 0.25)]
all_stats = {}

plt.figure(figsize=(12, 4))
for i, (p, q) in enumerate(walk_configs, 1):
    walks = [node2vec_walk(G, start=start_node, p=p, q=q, walk_length=20) for _ in range(500)]
    st = walk_stats(walks, G)
    all_stats[(p, q)] = st

    print(f"(p={p}, q={q}) -> return={st['return_frac']:.3f}, dist>=2={st['dist2plus_frac']:.3f}, avg unique={st['unique_counts'].mean():.2f}")

    plt.subplot(1, 3, i)
    plt.hist(st['unique_counts'], bins=12, alpha=0.8)
    plt.title(f"p={p}, q={q}")
    plt.xlabel('unique nodes/walk')
    plt.ylabel('count')

plt.tight_layout()
plt.show()

# Tests/checks
one_walk = node2vec_walk(G, start_node, p=1, q=1, walk_length=20)
assert 1 <= len(one_walk) <= 20, "Walk length bounds violated"
assert all(0.0 <= all_stats[c]['return_frac'] <= 1.0 for c in walk_configs), "Return fraction out of range"
assert all(0.0 <= all_stats[c]['dist2plus_frac'] <= 1.0 for c in walk_configs), "Distance-2+ fraction out of range"


The custom biased-walk implementation produces statistics consistent with node2vec intuition:

- \((p,q)=(0.25,4)\): return fraction **0.133**, dist\(\ge2\) fraction **0.324**, avg unique **16.61**
- \((1,1)\): return **0.019**, dist\(\ge2\) **0.667**, avg unique **19.04**
- \((4,0.25)\): return **0.002**, dist\(\ge2\) **0.878**, avg unique **19.48**

Interpretation:
- Lower **p** increases backtracking (higher return fraction), which reduces exploration diversity (fewer unique nodes).
- Lower **q** encourages outward moves to nodes farther from the previous node (higher dist\(\ge2\) fraction), increasing coverage (more unique nodes).

So the measured walk behavior aligns well with the expected roles of **p** (return control) and **q** (inward/outward exploration).



Why the tests support correctness:
- `assert 1 <= len(one_walk) <= 20` verifies walk generation always respects termination and maximum length constraints, so the sampler is not over-stepping or returning empty invalid walks.
- The two range assertions for `return_frac` and `dist2plus_frac` enforce that both metrics behave as valid probabilities in \([0,1]\). Since these values are computed by counting events over steps, a failure would indicate a bug in step counting, denominator handling, or the return/distance classification logic.
- Together, these assertions check both structural correctness of the walk output and semantic correctness of the derived statistics, which is exactly what Part B uses to interpret node2vec behavior.

# 4. LSH at Different Thresholds  [15pts]

Using `datasketch.MinHash` and `datasketch.MinHashLSH`, apply the LSH pipeline to the Congress Twitter Network. Treat each member's **set of followees** (outgoing neighbors) as their document. Use members with ≥ 10 followees and 128 permutations.

Run `MinHashLSH` at thresholds: **0.1, 0.2, 0.3, 0.5, 0.7**. For each threshold report:
- Number of candidate pairs returned
- Of a random sample of 50 candidate pairs, the fraction with true Jaccard ≥ threshold (precision proxy)
- Of the top-20 truly similar pairs by exact Jaccard, how many were recovered (recall proxy)

Present your results as a table, then:
1. Plot candidate pair count (y-axis, log scale) vs. threshold (x-axis). Describe the shape of the curve.
2. You need to recover all pairs with true Jaccard ≥ 0.4 without missing any. Which threshold do you choose, and what do you give up?
3. At threshold=0.1, is LSH still useful compared to brute force? Why or why not?

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

In [ ]:
# Q4: LSH over followee sets at multiple thresholds
from datasketch import MinHash, MinHashLSH

# Build followee sets for members with >=10 followees
followee_sets = {}
for n in node_list:
    outs = set(G.successors(n))
    if len(outs) >= 10:
        followee_sets[n] = outs

nodes_filtered = sorted(followee_sets.keys(), key=lambda x: int(x))
print(f"Members with >=10 followees: {len(nodes_filtered)}")

# MinHash signatures
num_perm = 128
signatures = {}
for n in nodes_filtered:
    mh = MinHash(num_perm=num_perm)
    for token in sorted(followee_sets[n]):
        mh.update(str(token).encode('utf8'))
    signatures[n] = mh

def exact_jaccard(a, b):
    A, B = followee_sets[a], followee_sets[b]
    return len(A & B) / len(A | B)

# Ground-truth exact top-20 similar pairs
all_pairs = []
for i in range(len(nodes_filtered)):
    for j in range(i + 1, len(nodes_filtered)):
        u, v = nodes_filtered[i], nodes_filtered[j]
        all_pairs.append((u, v, exact_jaccard(u, v)))
all_pairs_sorted = sorted(all_pairs, key=lambda x: x[2], reverse=True)
true_top20 = {(u, v) for u, v, _ in all_pairs_sorted[:20]}

thresholds = [0.1, 0.2, 0.3, 0.5, 0.7]
results = []

for th in thresholds:
    lsh = MinHashLSH(threshold=th, num_perm=num_perm)
    for n in nodes_filtered:
        lsh.insert(n, signatures[n])

    candidate_pairs = set()
    for n in nodes_filtered:
        for m in lsh.query(signatures[n]):
            if m == n:
                continue
            a, b = sorted([n, m], key=lambda x: int(x))
            candidate_pairs.add((a, b))

    cand_list = sorted(candidate_pairs, key=lambda x: (int(x[0]), int(x[1])))
    sample_size = min(50, len(cand_list))
    sample_pairs = random.sample(cand_list, sample_size) if sample_size > 0 else []

    if sample_size > 0:
        good = sum(1 for (a, b) in sample_pairs if exact_jaccard(a, b) >= th)
        precision_proxy = good / sample_size
    else:
        precision_proxy = np.nan

    recovered = len(true_top20 & set(candidate_pairs))

    results.append({
        'threshold': th,
        'candidate_pairs': len(candidate_pairs),
        'precision_proxy': precision_proxy,
        'recall_proxy_top20': recovered,
    })

results_df = pd.DataFrame(results)
display(results_df)

# Plot count vs threshold (log y-scale)
plt.figure(figsize=(6, 4))
plt.plot(results_df['threshold'], results_df['candidate_pairs'], marker='o')
plt.yscale('log')
plt.xlabel('LSH threshold')
plt.ylabel('Candidate pairs (log scale)')
plt.title('Candidate pairs vs threshold')
plt.grid(alpha=0.3)
plt.show()

# Optional diagnostic for >=0.4 target coverage
true_ge_04 = {(u, v) for (u, v, jac) in all_pairs if jac >= 0.4}
coverage = []
for th in thresholds:
    lsh = MinHashLSH(threshold=th, num_perm=num_perm)
    for n in nodes_filtered:
        lsh.insert(n, signatures[n])
    cp = set()
    for n in nodes_filtered:
        for m in lsh.query(signatures[n]):
            if m == n:
                continue
            a, b = sorted([n, m], key=lambda x: int(x))
            cp.add((a, b))
    cov = len(true_ge_04 & cp) / max(len(true_ge_04), 1)
    coverage.append((th, cov))
print("Coverage of true Jaccard>=0.4 pairs by threshold:", coverage)

# Tests/checks
assert len(nodes_filtered) > 0 and len(signatures) == len(nodes_filtered), "Signature build failed"
assert all(0 <= r['precision_proxy'] <= 1 or np.isnan(r['precision_proxy']) for r in results), "Precision proxy out of range"
assert all(results_df['candidate_pairs'].iloc[i] >= results_df['candidate_pairs'].iloc[i+1] for i in range(len(results_df)-1)), "Candidate pairs should generally decrease as threshold increases"


The LSH threshold sweep shows a clear precision/recall-vs-efficiency tradeoff.

- Candidate pairs drop sharply as threshold increases: **19,722 (0.1)** → **11,404 (0.2)** → **3,122 (0.3)** → **127 (0.5)** → **1 (0.7)**.
- Recall proxy on the exact top-20 pairs is high at low thresholds (**20, 20, 19**) and then falls (**8, 1**).
- Coverage of true Jaccard \(\ge 0.4\) pairs is **1.0** at thresholds 0.1 and 0.2, **0.955** at 0.3, then drops to **0.409** at 0.5.

A practical choice here is around **threshold 0.2–0.3**: it keeps most high-similarity pairs while reducing candidate volume substantially compared with 0.1. Higher thresholds become too aggressive and miss many true similar pairs.

Why the tests support correctness:
- `assert len(nodes_filtered) > 0 and len(signatures) == len(nodes_filtered)` validates that every eligible node receives a MinHash signature; without this, candidate and recall calculations could silently exclude nodes.
- The precision-proxy range assertion (`0 <= precision_proxy <= 1` or NaN when no candidates) confirms the metric computation is numerically valid and properly handles edge cases.
- The monotonic candidate-pair assertion across thresholds checks the core intended behavior of LSH thresholding: stricter thresholds should not produce more candidates. If this fails, insertion/query logic or threshold application is likely incorrect.
- These checks collectively verify data pipeline integrity, metric correctness, and expected algorithmic behavior, which is why the resulting tradeoff conclusions are trustworthy.



# Resources for Section A

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# B [30pts]. Interview Questions

We now pretend this is a real job interview. Here's some guidance on how to answer these questions:

1. Briefly restate the question and state any assumptions you are making.
2. Explain your reasoning out loud, focusing on tradeoffs, limitations, and constraints.
3. Keep answers as short and clear as they can be (while still answering the question).
4. Write/speak in a conversational but professional tone.

There may not be a single correct answer. We are grading whether your reasoning is reasonable and aware of limitations.

**Rubric**

[10pt] Clear understanding of the question; reasonable assumptions; thoughtful reasoning that acknowledges tradeoffs and limitations; clear, concise communication.

[5pt] Basic understanding but shallow reasoning or unclear assumptions.

[0pt] Minimal, unclear, or incorrect response.

# 1.
We need graph embeddings for a 'People You May Know' feature — should we use DeepWalk, LINE, or node2vec? What's actually different about them?

*your answer here*

# 2.
We're building a near-duplicate detection system for 50 million documents. Why would you use LSH over just comparing all pairs — especially if LSH can miss things?

*your answer here*

# 3.
(Video) Explain node2vec to me. Specifically — why does the p/q biasing actually matter? Isn't a random walk just a random walk?

*your answer here (include video link)*

# Resources for Section B

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# C [7pts]. What new questions do you have?
We want you to think bigger! Tell us what questions and curiosity this homework brings up for you.

**Rubric**

[7pt] Complete, thoughtful response.

[4pt] Partial response.

[0pt] Minimal response.

# 1.
What new questions do you have after this homework? Or, what topics are you curious about now? List at least 3.

*your answer here*